# Instructor-Notebook (nicht für Studierende)

**Achtung: Dieses Notebook zeigt die Lösung (Ground Truth).**

Vorbereitung und Betrieb der Session: Ground Truth hochladen, Mock-Durchlauf, Vorab-Läufe als Backup, Baseline auf dem Leaderboard, Auflösung, Sicherungen nachsenden, Kosten prüfen. Ablauf und Fallbacks: `docs/RUNBOOK.md`.

In [ ]:
#@title ⚙️ Setup

import os
import subprocess
import sys

REPO_URL = "https://github.com/evabowe/software-quality-engineering"
# Lokal (Jupyter im Repo) wird der vorhandene Checkout genutzt, in Colab ein frischer Klon.
if os.path.isfile("../pyproject.toml"):
    REPO_DIR = os.path.abspath("..")
else:
    REPO_DIR = "/content/software-quality-engineering"
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)  # die Installation wird sonst erst nach einem Neustart sichtbar

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

from getpass import getpass
from pathlib import Path

from IPython.display import display

from judge_lab import config, instructor
from judge_lab.llm import MockClient, make_client
from judge_lab.runner import RunContext, run_challenge, run_matrix
from judge_lab.scoreboard import Scoreboard
from judge_lab.snippets import load_snippets

API_KEY = getpass("Instructor-Key für OpenRouter: ")
board = Scoreboard(config.SCOREBOARD_URL, config.SCOREBOARD_TOKEN)
snippets = load_snippets()
print("Scoreboard:", config.SCOREBOARD_URL or "(nicht konfiguriert – nur CSV)", "· Session:", config.DEFAULT_SESSION_ID)

In [ ]:
#@title 📋 Ground Truth anzeigen und ans Scoreboard senden
import pandas as pd

display(pd.DataFrame(instructor.ground_truth_rows()))
if config.SCOREBOARD_URL:
    ok = instructor.upload_ground_truth(config.SCOREBOARD_URL, config.SCOREBOARD_TOKEN)
    print("✅ hochgeladen" if ok else "⚠️ Upload fehlgeschlagen (Token? URL?)")

In [ ]:
#@title 🧪 Mock-Durchlauf (Session „ci“, keine Kosten)
ctx = RunContext(group="Gruppe 1", model_key="mock", session_id="ci", sink=board.submit)
mock_records = run_matrix(ctx, MockClient(snippets), snippets, config.ATTITUDES, (True, False))
print(len(mock_records), "Läufe gesendet – Dashboard mit ?session=ci prüfen")

In [ ]:
#@title 💾 Vorab-Läufe aller Modelle (Session „precomputed“, kostet Tokens)
MODEL_KEYS = ["gemini", "deepseek", "kimi", "local_fallback"]
precomputed = instructor.precompute(MODEL_KEYS, API_KEY, board, on_progress=lambda d, t: print(f"\r{d}/{t}", end=""))
display(instructor.resolution_table(precomputed))

In [ ]:
#@title 🏁 Baseline-Prompt auf dem Leaderboard (Gruppe „Baseline“, Referenzmodell)
baseline = (config.PROMPTS_DIR / "challenge_baseline.md").read_text(encoding="utf-8")
ctx = RunContext(group="Baseline", model_key="reference", session_id=config.DEFAULT_SESSION_ID, sink=board.submit)
reference = make_client("reference", API_KEY)
baseline_records = run_challenge(ctx, {"group": reference, "reference": reference}, baseline,
                                 snippets=snippets, submit=True, submission_no=1)
display(instructor.resolution_table(baseline_records))

In [ ]:
#@title 📤 Sicherung nachsenden (CSV aus einem Gruppen-Notebook)
BACKUP_CSV = "/content/scoreboard_backup.csv"  #@param {type:"string"}
sent = Scoreboard(config.SCOREBOARD_URL, config.SCOREBOARD_TOKEN, Path(BACKUP_CSV)).replay_backup()
print(sent, "Läufe nachgesendet")

In [ ]:
#@title 💶 Kosten (OpenRouter-Key)
KEY_TO_CHECK = ""  #@param {type:"string"}
print(instructor.openrouter_usage(KEY_TO_CHECK or API_KEY))